# Re-running the core analyses with the learned subgroup embedding (representation C)

`feature_ablation_multiseed.ipynb` showed that the learned, count-weighted subgroup embedding (C) generalizes
best. On full BIMOG it scores 0.161 ± 0.008, against 0.307 for hand-crafted descriptors (B) and 0.257 for
unscaled raw counts. On MCM it scores 0.031, against 0.098 for B and 0.078 for the GNN. C is therefore adopted
as the main surrogate, and the analyses that depend on the surrogate representation are repeated with it:

1. Table 1, training-pool composition (5 seeds)
2. Table 2, Model A in the Part II configuration (5 seeds)
3. Sect. 3.3, coverage analysis by five-fold CV (3 seeds)
4. Sect. 3.5, MCM learning curves and transfer (3 seeds)
5. BAT comparison on the same held-out points
6. Sect. 3.9, literature compounds incl. malonic acid (M1–M4, 3 seeds)
7. Sect. 3.6, viscosity emulator with C as the structural input (3 seeds)

**Model C.** Subgroup counts (unscaled) → bias-free linear map to 16 dimensions (= sum of learned
per-subgroup embeddings weighted by count) → concatenated with standardized extra inputs (w or a_w, T,
optionally ln T_g) → 3 × 128 ReLU head. The training loop is unchanged from Part I.

In [1]:
import json, time, hashlib
import numpy as np, pandas as pd, torch, torch.nn as nn
from scipy import stats
from rdkit import Chem, RDLogger
from aiomfac_py import ActivityModel, Component
from aiomfac_py.s2as import smiles_to_components
from aiomfac_py.params import load_subgroup_params
import aiomfac_py.s2as._mapping as _m
_m.print = lambda *a, **k: None; RDLogger.DisableLog("rdApp.*")
sg = load_subgroup_params(); WATER = Component(1, "Water", ((16, 1),))
hash100 = lambda s: int(hashlib.sha256(s.encode()).hexdigest(), 16) % 100
fmt = lambda s: f"{np.mean(s):.3f} ± {np.std(s, ddof=1):.3f}" if len(s) > 1 else f"{np.mean(s):.3f}"
T0 = time.time()

def featurize(name, smi, cat):
    r = smiles_to_components([smi], water_as_component1=True)
    if r.removed: return None
    try:
        if not (ActivityModel(r.components).evaluate([0.9, 0.1], 298.15, basis="mass").gamma_neutral > 0).all(): return None
    except Exception: return None
    mol = Chem.MolFromSmiles(smi); mh = Chem.AddHs(mol); sym = [a.GetSymbol() for a in mh.GetAtoms()]; nc = max(sym.count("C"), 1)
    return {"name": name, "smiles": smi, "canon": Chem.MolToSmiles(mol), "cat": cat, "subgroups": r.components[1].subgroups,
            "M": sum(a.GetMass() for a in mh.GetAtoms()), "oc": sym.count("O") / nc, "nc": sym.count("N") / nc,
            "hc": sym.count("H") / nc, "unm": int(r.n_unmatched_atoms[0])}

def bimog_cat(d):
    n, h = d["#N"] != 0, d["#Hal"] != 0
    return "n_and_hal" if n and h else "n_only" if n else "hal_only" if h else "cho"

bim_raw = json.load(open("bimog_unified.json"))
bimog = [o for d in bim_raw if (o := featurize(d["name"], d["smiles"], bimog_cat(d)))]
bcanon = {o["canon"] for o in bimog}

synth = []
add = lambda s, n: synth.append((n, s))
for n in range(1, 11): add("C" * n + "N", f"amine1-C{n}")
for n in range(1, 7): add("C" * n + "N" + "C" * n, f"amine2s-C{n}")
for n in range(1, 6):
    for m in range(n + 1, 7): add("C" * n + "N" + "C" * m, f"amine2-C{n}C{m}")
for n in range(1, 9): add("C" * n + "[N+](=O)[O-]", f"nitro-C{n}")
add("c1ccccc1[N+](=O)[O-]", "nitrobenzene")
for n in range(1, 5): add("C" * n + "c1ccccc1[N+](=O)[O-]", f"nitroarom-C{n}")
for n in range(1, 11): add("C" * n + "O[N+](=O)[O-]", f"nitrate-C{n}")
for n in range(2, 8):
    add("C" * (n - 1) + "C(O)O[N+](=O)[O-]", f"hn-a-C{n}"); add("OC" + "C" * (n - 2) + "CO[N+](=O)[O-]", f"hn-b-C{n}")
for n in range(2, 8): add("NC" + "C" * (n - 2) + "CO", f"aminoalc-C{n}")
for s, n in [("CC(C)N", "isopropylamine"), ("CC(C)(C)N", "tBuNH2"), ("CC(C)CN", "iBuNH2"), ("CC(N)CC", "sBuNH2"),
             ("C1CCC(CC1)N", "cyhexNH2"), ("C1CCC(C1)N", "cypentNH2"), ("CC(C)(C)CN", "neopentNH2"), ("COCCN", "MeOEtNH2"),
             ("CCOCCCN", "EtOPrNH2"), ("CC(=O)CCN", "aminobutanone"), ("CCOC(=O)CCN", "Et-aminopropanoate"),
             ("NCCOCCN", "bisaminoethylether"), ("NCCCCN", "putrescine"), ("NCCCCCN", "cadaverine"), ("NCCCCCCN", "HMDA"),
             ("NC(C)CN", "diaminopropane"), ("CC(C)[N+](=O)[O-]", "2-nitropropane"), ("C1CCC(CC1)[N+](=O)[O-]", "nitrocyclohexane"),
             ("CC(C)(C)[N+](=O)[O-]", "tBuNO2"), ("CC(=O)CC[N+](=O)[O-]", "nitrobutanone"), ("COCC[N+](=O)[O-]", "MeO-nitroethane"),
             ("CC(=O)CO[N+](=O)[O-]", "nitrooxypropanone"), ("COCCO[N+](=O)[O-]", "MeOEt-nitrate"), ("CC(C)O[N+](=O)[O-]", "iPr-nitrate"),
             ("C1CCC(CC1)O[N+](=O)[O-]", "cyhex-nitrate"), ("CC(C)NC(C)C", "diisopropylamine"), ("C1CCNCC1", "piperidine"),
             ("C1CCNC1", "pyrrolidine")]:
    add(s, n)
synth = [o for n, s in synth if (o := featurize(n, s, "synth_n"))]

mraw = pd.read_csv("mcm_v331_species.tsv", sep="\t", comment="*"); mcm = []
for _, r in mraw.iterrows():
    mol = Chem.MolFromSmiles(str(r.Smiles))
    if (mol is None or str(r.Excited).lower() == "true" or any(a.GetNumRadicalElectrons() for a in mol.GetAtoms())
            or not {a.GetSymbol() for a in mol.GetAtoms()} <= {"C", "H", "O", "N"} or mol.GetNumHeavyAtoms() < 2
            or "C" not in {a.GetSymbol() for a in mol.GetAtoms()}): continue
    o = featurize(r.Name, str(r.Smiles), "mcm")
    if o and o["unm"] == 0 and o["canon"] not in bcanon: mcm.append(o)

POOL = bimog + synth + mcm
SRC = np.array(["bimog"] * len(bimog) + ["synth"] * len(synth) + ["mcm"] * len(mcm))
CAT = np.array([o["cat"] for o in POOL]); H = np.array([hash100(o["smiles"]) for o in POOL]); TEST = H < 15
VOCAB = sorted({int(s) for o in POOL for s, q in o["subgroups"]})
def counts(o):
    v = np.zeros(len(VOCAB), np.float32)
    for s, q in o["subgroups"]: v[VOCAB.index(int(s))] = q
    return v
CNT = np.array([counts(o) for o in POOL])
print(f"BIMOG {len(bimog)}, synthetic N {len(synth)}, MCM {len(mcm)}; vocabulary {len(VOCAB)} subgroups ({time.time()-T0:.0f}s)")

BIMOG 368, synthetic N 100, MCM 3337; vocabulary 55 subgroups (10s)


In [2]:
def label(ids, n_comp=20, n_temp=4, seed=0):
    '''AIOMFAC labels for the molecules in ids (Part I sampling). Returns molecule index, (w, T_norm), y.'''
    rng = np.random.RandomState(seed); rows = []
    for oi in ids:
        m = ActivityModel([WATER, Component(2, "o", tuple(POOL[oi]["subgroups"]))]); e = np.linspace(0.02, 0.98, n_comp + 1)
        for w in rng.uniform(e[:-1], e[1:]):
            for T in rng.uniform(273.15, 313.15, n_temp):
                try:
                    lg = m.evaluate([w, 1 - w], T, basis="mass").ln_gamma
                    if np.isfinite(lg[:2]).all(): rows.append((oi, w, (T - 293.15) / 20, lg[0], lg[1]))
                except Exception: pass
    a = np.array(rows)
    return a[:, 0].astype(int), a[:, 1:3].astype(np.float32), a[:, 3:5].astype(np.float32)

PM, EXTRA, Y = label(np.arange(len(POOL)))                     # 20 x 4 labels for every molecule
print(f"{len(Y)} labelled points ({time.time()-T0:.0f}s)")


class EmbedNet(nn.Module):
    def __init__(self, n_vocab, n_extra, n_out=2, emb=16):
        super().__init__()
        self.n_vocab = n_vocab; self.emb = nn.Linear(n_vocab, emb, bias=False)
        self.head = nn.Sequential(nn.Linear(emb + n_extra, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU(),
                                  nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, n_out))
    def forward(self, x): return self.head(torch.cat([self.emb(x[:, :self.n_vocab]), x[:, self.n_vocab:]], 1))


def train_C(Xc, Xe, Yt, train_pts, seed, interp_split=False, max_epochs=500, patience=30, batch=256):
    '''Xc: per-point counts (unscaled); Xe: per-point extra inputs (standardized here). Returns predict(Xc, Xe).'''
    tr = np.array(train_pts)
    if interp_split:                                     # Part I/II: fixed 20 % interpolation split, val = first 10 %
        p = np.random.RandomState(0).permutation(len(tr)); fit = tr[p[int(0.2 * len(tr)):]]
        nv = int(0.1 * len(fit)); val, fit = fit[:nv], fit[nv:]
    else:                                                # learning-curve style: seeded 10 % validation
        r0 = np.random.RandomState(seed); tr = tr.copy(); r0.shuffle(tr); nv = int(0.1 * len(tr)); val, fit = tr[:nv], tr[nv:]
    em, es = Xe[fit].mean(0), Xe[fit].std(0); es = np.where(es < 1e-6, 1.0, es)
    ym, ys = Yt[fit].mean(0), Yt[fit].std(0) + 1e-6
    X = torch.tensor(np.concatenate([Xc, (Xe - em) / es], 1).astype(np.float32)); Yn = torch.tensor((Yt - ym) / ys)
    torch.manual_seed(seed); rng = np.random.RandomState(seed)
    net = EmbedNet(Xc.shape[1], Xe.shape[1], Yt.shape[1]); opt = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-5)
    L = nn.MSELoss(); best, st, bad = 1e9, None, 0; fit_t, val_t = torch.tensor(fit), torch.tensor(val)
    for ep in range(max_epochs):
        net.train(); perm = fit_t[torch.from_numpy(rng.permutation(len(fit)))]
        for s in range(0, len(perm), batch):
            i = perm[s:s + batch]; opt.zero_grad(); L(net(X[i]), Yn[i]).backward(); opt.step()
        net.eval()
        with torch.no_grad(): v = L(net(X[val_t]), Yn[val_t]).item()
        if v < best - 1e-5: best, st, bad = v, {k: t.clone() for k, t in net.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= patience: break
    net.load_state_dict(st); net.eval()
    def predict(xc, xe):
        with torch.no_grad():
            return net(torch.tensor(np.concatenate([xc, (xe - em) / es], 1).astype(np.float32))).numpy() * ys + ym
    return predict, ep + 1


def pts_of(mask_mol, pm=None):
    pm = PM if pm is None else pm
    return np.where(mask_mol[pm])[0]

304400 labelled points (34s)


## 1) Table 1: training-pool composition (5 seeds, Part I configuration)

In [3]:
ids = np.arange(len(POOL)); isB = SRC == "bimog"
POOLS = {"A: CHO only": isB & (CAT == "cho"), "B: full BIMOG": isB, "C: B + synthetic N": isB | (SRC == "synth")}
t1 = []
for pname, pmask in POOLS.items():
    train_m, test_m = pmask & ~TEST, isB & TEST          # held-out set: the same BIMOG molecules for every pool
    for seed in range(5):
        pred, ep = train_C(CNT[PM], EXTRA, Y, pts_of(train_m), seed, interp_split=True)
        g = pts_of(test_m); e = np.abs(pred(CNT[PM[g]], EXTRA[g]) - Y[g])
        rec = {"pool": pname, "seed": seed, "gen_w": e[:, 0].mean(), "gen_org": e[:, 1].mean(), "epochs": ep}
        for c in ["cho", "n_only", "hal_only", "n_and_hal"]:
            sel = CAT[PM[g]] == c
            if sel.any(): rec[f"gen_org_{c}"] = e[sel, 1].mean()
        t1.append(rec)
    print(f"  {pname} done ({time.time()-T0:.0f}s)")
t1 = pd.DataFrame(t1); t1.to_csv("C_table1_runs.csv", index=False)
cols = ["gen_w", "gen_org", "gen_org_cho", "gen_org_n_only", "gen_org_hal_only", "gen_org_n_and_hal"]
display(pd.DataFrame({p: {c: fmt(g[c].dropna()) if g[c].notna().any() else "–" for c in cols} for p, g in t1.groupby("pool", sort=False)}).T)

  A: CHO only done (66s)


  B: full BIMOG done (113s)


  C: B + synthetic N done (180s)


,gen_w,gen_org,gen_org_cho,gen_org_n_only,gen_org_hal_only,gen_org_n_and_hal
A: CHO only,0.038 ± 0.002,0.370 ± 0.021,0.181 ± 0.035,0.714 ± 0.044,0.417 ± 0.174,0.588 ± 0.050
B: full BIMOG,0.015 ± 0.001,0.147 ± 0.016,0.119 ± 0.019,0.186 ± 0.015,0.208 ± 0.077,0.181 ± 0.023
C: B + synthetic N,0.016 ± 0.002,0.148 ± 0.015,0.120 ± 0.021,0.193 ± 0.023,0.188 ± 0.025,0.175 ± 0.028


## 2) Table 2, Model A with representation C (Part II configuration: 10 w × 3 T, 300 epochs, patience 25)

In [4]:
bids = np.where(isB)[0]
PM2, EX2, Y2 = label(bids, n_comp=10, n_temp=3)
t2 = []
for seed in range(5):
    pred, ep = train_C(CNT[PM2], EX2, Y2, np.where(~TEST[PM2])[0], seed, interp_split=True, max_epochs=300, patience=25)
    g = np.where(TEST[PM2])[0]; e = np.abs(pred(CNT[PM2[g]], EX2[g]) - Y2[g])
    t2.append({"seed": seed, "gen_w": e[:, 0].mean(), "gen_org": e[:, 1].mean(), "epochs": ep})
t2 = pd.DataFrame(t2); t2.to_csv("C_table2_modelA_runs.csv", index=False)
print("Model A (representation C), Part II configuration:", "water", fmt(t2.gen_w), "| organic", fmt(t2.gen_org))

Model A (representation C), Part II configuration: water 0.021 ± 0.001 | organic 0.181 ± 0.009


## 3) Coverage analysis (Sect. 3.3): five-fold CV over all 368 BIMOG molecules, 3 seeds

In [5]:
fold = H % 5
oof = {}
for seed in range(3):
    for k in range(5):
        trm, tem = isB & (fold != k), isB & (fold == k)
        pred, _ = train_C(CNT[PM], EXTRA, Y, pts_of(trm), 1000 * seed + k)
        g = pts_of(tem); e = np.abs(pred(CNT[PM[g]], EXTRA[g]) - Y[g])[:, 1]
        for oi, v in pd.Series(e).groupby(PM[g]).mean().items():
            oof.setdefault(oi, []).append(v)
    print(f"  seed {seed} done ({time.time()-T0:.0f}s)")
per = pd.DataFrame({"mol": list(oof), "err": [np.mean(v) for v in oof.values()]})
per["cat"] = CAT[per.mol]; per["unmatched"] = [POOL[m]["unm"] > 0 for m in per.mol]
print("Out-of-fold MAE ln γ_org (mean over 3 seeds per molecule):")
display(per.groupby("cat").err.agg(["mean", "count"]).round(3)); display(per.groupby("unmatched").err.agg(["mean", "count"]).round(3))
# novelty control: distance to nearest other-fold molecule in standardized count space, and label magnitude
Z = CNT[bids]; Z = (Z - Z.mean(0)) / np.where(Z.std(0) < 1e-6, 1, Z.std(0))
pos = {m: i for i, m in enumerate(bids)}
per["nn"] = [np.min(np.linalg.norm(Z[[pos[j] for j in bids if fold[j] != fold[m]]] - Z[pos[m]], axis=1)) for m in per.mol]
per["mag"] = [np.abs(Y[PM == m, 1]).mean() for m in per.mol]
A_ = np.column_stack([np.ones(len(per)), per.nn, per.mag, per.unmatched.astype(float)]); y_ = per.err.values
beta = np.linalg.lstsq(A_, y_, rcond=None)[0]; rs = np.random.RandomState(0)
boots = np.array([np.linalg.lstsq(A_[i], y_[i], rcond=None)[0] for i in [rs.randint(0, len(y_), len(y_)) for _ in range(2000)]])
lo, hi = np.percentile(boots, [2.5, 97.5], 0)
print(f"error = {beta[0]:.3f} + {beta[1]:.3f}·nn [{lo[1]:.3f},{hi[1]:.3f}] + {beta[2]:.3f}·|lnγ| [{lo[2]:.3f},{hi[2]:.3f}] "
      f"+ {beta[3]:+.3f}·unmatched [{lo[3]:+.3f},{hi[3]:+.3f}]")
per.to_csv("C_coverage_oof.csv", index=False)

  seed 0 done (247s)


  seed 1 done (289s)


  seed 2 done (332s)
Out-of-fold MAE ln γ_org (mean over 3 seeds per molecule):


,mean,count
cat,,
cho,0.139,243
hal_only,0.132,22
n_and_hal,0.155,29
n_only,0.204,74


,mean,count
unmatched,,
False,0.143,245
True,0.173,123


error = -0.016 + 0.051·nn [0.032,0.076] + 0.014·|lnγ| [0.001,0.026] + -0.012·unmatched [-0.053,+0.031]


## 4) MCM learning curves and transfer (3 seeds)

In [6]:
mcm_tr = np.where((SRC == "mcm") & ~TEST)[0]; mcm_te_mask = (SRC == "mcm") & TEST
b_tr = np.where(isB & ~TEST)[0]; b_te_mask = isB & TEST
lc = []
for seed in range(3):
    order = np.random.RandomState(100 + seed).permutation(mcm_tr)
    configs = [("MCM only", order[:n]) for n in (100, 200, 400, 800, 1600, len(order))] + \
              [("BIMOG train + MCM", np.r_[b_tr, order[:n]]) for n in (0, 400, 1600, len(order))]
    for reg, sub in configs:
        m = np.zeros(len(POOL), bool); m[sub] = True
        pred, ep = train_C(CNT[PM], EXTRA, Y, pts_of(m), seed, max_epochs=300)
        rec = {"regime": reg, "n_train_mol": len(sub), "seed": seed, "epochs": ep}
        for tname, tmask in [("MCM", mcm_te_mask), ("BIMOG", b_te_mask)]:
            g = pts_of(tmask); e = np.abs(pred(CNT[PM[g]], EXTRA[g]) - Y[g]); rec[f"{tname} org"] = e[:, 1].mean(); rec[f"{tname} w"] = e[:, 0].mean()
        lc.append(rec)
        print(f"  seed {seed} {reg:18s} n={len(sub):5d}: MCM {rec['MCM org']:.3f}  BIMOG {rec['BIMOG org']:.3f} ({time.time()-T0:.0f}s)")
lc = pd.DataFrame(lc); lc.to_csv("C_learning_curve_runs.csv", index=False)
display(lc.groupby(["regime", "n_train_mol"])[["MCM org", "BIMOG org", "MCM w"]].agg(fmt))

  seed 0 MCM only           n=  100: MCM 0.252  BIMOG 1.601 (336s)


  seed 0 MCM only           n=  200: MCM 0.191  BIMOG 1.363 (347s)


  seed 0 MCM only           n=  400: MCM 0.103  BIMOG 1.294 (359s)


  seed 0 MCM only           n=  800: MCM 0.055  BIMOG 0.781 (376s)


  seed 0 MCM only           n= 1600: MCM 0.037  BIMOG 0.680 (404s)


...

  seed 2 BIMOG train + MCM  n=  312: MCM 0.534  BIMOG 0.152 (1170s)


  seed 2 BIMOG train + MCM  n=  712: MCM 0.083  BIMOG 0.118 (1191s)


  seed 2 BIMOG train + MCM  n= 1912: MCM 0.041  BIMOG 0.094 (1249s)


  seed 2 BIMOG train + MCM  n= 3167: MCM 0.031  BIMOG 0.092 (1358s)


MCM org      BIMOG org          MCM w
regime            n_train_mol                                             
BIMOG train + MCM 312          0.529 ± 0.004  0.148 ± 0.007  0.104 ± 0.001
                  712          0.082 ± 0.003  0.127 ± 0.009  0.012 ± 0.000
                  1912         0.041 ± 0.003  0.099 ± 0.005  0.007 ± 0.000
                  3167         0.034 ± 0.002  0.102 ± 0.012  0.007 ± 0.000
MCM only          100          0.253 ± 0.002  1.615 ± 0.014  0.034 ± 0.001
                  200          0.163 ± 0.024  1.352 ± 0.125  0.023 ± 0.001
                  400          0.095 ± 0.007  1.091 ± 0.177  0.015 ± 0.000
                  800          0.053 ± 0.004  0.806 ± 0.042  0.009 ± 0.001
                  1600         0.037 ± 0.002  0.697 ± 0.015  0.007 ± 0.000
                  2855         0.030 ± 0.001  0.672 ± 0.028  0.006 ± 0.000

## 5) BAT on the same held-out BIMOG points as Table 1, pool B

In [7]:
from particula.activity import bat_activity_coefficients
from particula.particles import get_organic_density_array
g = pts_of(isB & TEST); oi = PM[g]
M = np.array([POOL[i]["M"] for i in oi]); oc = np.array([POOL[i]["oc"] for i in oi])
hc = np.array([POOL[i]["hc"] for i in oi]); ncr = np.array([POOL[i]["nc"] for i in oi]); w = EXTRA[g, 0]
xo = ((1 - w) / M) / ((1 - w) / M + w / 18.015)
out = bat_activity_coefficients(18.015 / M, xo, oc, get_organic_density_array(M, oc, hc, ncr))
e_bat = np.abs(np.column_stack([np.log(out[4]), np.log(out[5])]) - Y[g])
e_c = np.mean([np.abs(train_C(CNT[PM], EXTRA, Y, pts_of(isB & ~TEST), s, interp_split=True)[0](CNT[PM[g]], EXTRA[g]) - Y[g]) for s in range(5)], 0)
cho = CAT[oi] == "cho"
pm_c = pd.Series(e_c[cho, 1]).groupby(oi[cho]).mean(); pm_b = pd.Series(e_bat[cho, 1]).groupby(oi[cho]).mean()
print(f"all held-out: C org {e_c[:,1].mean():.3f} w {e_c[:,0].mean():.3f} | BAT org {e_bat[:,1].mean():.3f} w {e_bat[:,0].mean():.3f}")
print(f"CHO held-out ({cho.sum()} pts, {len(pm_c)} mol): C {e_c[cho,1].mean():.3f} vs BAT {e_bat[cho,1].mean():.3f}; C better for {(pm_c < pm_b).sum()}/{len(pm_c)} molecules")

all held-out: C org 0.147 w 0.015 | BAT org 1.387 w 0.128
CHO held-out (2640 pts, 33 mol): C 0.119 vs BAT 1.025; C better for 33/33 molecules


## 6) Literature compounds (Sect. 3.9): M1–M4 with representation C (3 seeds)

In [8]:
TARGETS = {"Glycerol": "OCC(O)CO", "1,2,4-Butanetriol": "OCCC(O)CO", "2,5-Hexanediol": "CC(O)CCC(C)O",
           "Ethanol": "CCO", "2-Propanol": "CC(C)O", "Malonic acid": "OC(=O)CC(=O)O"}
W19 = np.linspace(0.05, 0.95, 19); truth, Q = {}, {}
for nm, s in TARGETS.items():
    o = featurize(nm, s, "t"); m = ActivityModel([WATER, Component(2, "o", tuple(o["subgroups"]))])
    truth[nm] = np.array([m.evaluate([x, 1 - x], 298.15, basis="mass").ln_gamma[:2] for x in W19])
    Q[nm] = (np.tile(counts(o), (19, 1)), np.column_stack([W19, np.full(19, (298.15 - 293.15) / 20)]).astype(np.float32))
cooh = Chem.MolFromSmarts("C(=O)[OH]")
diacid = np.array([SRC[i] == "mcm" and len(Chem.MolFromSmiles(POOL[i]["smiles"]).GetSubstructMatches(cooh)) >= 2 for i in range(len(POOL))])
isM = SRC == "mcm"
MODELS = {"M1": isB & ~TEST, "M2": (isB | isM) & ~TEST, "M3": isB | isM, "M4": (isB | isM) & ~TEST & ~diacid}
lit, lit_pred = [], {}
for mn, mask in MODELS.items():
    for seed in range(3):
        pred, _ = train_C(CNT[PM], EXTRA, Y, pts_of(mask), seed, max_epochs=300)
        for nm in TARGETS:
            p = pred(*Q[nm]); e = np.abs(p - truth[nm]); lit_pred[(mn, seed, nm)] = p
            lit.append({"model": mn, "seed": seed, "compound": nm, "w": e[:, 0].mean(), "org": e[:, 1].mean(), "max": e.max()})
    print(f"  {mn} done ({time.time()-T0:.0f}s)")
lit = pd.DataFrame(lit); lit.to_csv("C_literature_runs.csv", index=False)
display(lit.groupby(["compound", "model"], sort=False).agg(w=("w", "mean"), org=("org", "mean"), org_sd=("org", "std")).round(3).unstack("model"))
np.savez_compressed("C_literature_curves.npz", W19=W19, **{f"truth|{k}": v for k, v in truth.items()},
                    **{f"{mn}|{s}|{nm}": v for (mn, s, nm), v in lit_pred.items()})

  M1 done (1446s)


  M2 done (1699s)


  M3 done (1942s)


  M4 done (2157s)


w                         org                       \
model                 M1     M2     M3     M4     M1     M2     M3     M4   
compound                                                                    
Glycerol           0.006  0.006  0.005  0.007  0.031  0.022  0.034  0.052   
1,2,4-Butanetriol  0.004  0.004  0.005  0.004  0.015  0.018  0.017  0.036   
2,5-Hexanediol     0.005  0.004  0.003  0.002  0.025  0.022  0.022  0.026   
Ethanol            0.008  0.005  0.006  0.006  0.029  0.025  0.025  0.056   
2-Propanol         0.005  0.005  0.006  0.005  0.029  0.033  0.022  0.034   
Malonic acid       0.022  0.010  0.008  0.011  0.061  0.057  0.057  0.035   

                  org_sd                       
model                 M1     M2     M3     M4  
compound                                       
Glycerol           0.016  0.004  0.008  0.020  
1,2,4-Butanetriol  0.004  0.003  0.005  0.007  
2,5-Hexanediol     0.002  0.018  0.013  0.005  
Ethanol            0.006  0.014  0.009  0.008  
2-Propanol         0.004  0.006  0.007  0.005  
Malonic acid       0.018  0.013  0.007  0.005

## 7) Viscosity emulator (Sect. 3.6) with representation C as the structural input

In [9]:
from aiomfac_py.tgml_armeli import predict_tg_ml_smiles
from aiomfac_py.viscosity import organic_mixture_viscosity, pure_organic_viscosity_vtf
vm, vr = [], []
for d in bim_raw:
    try: tg = predict_tg_ml_smiles(d["smiles"]).tg_K
    except Exception: continue
    r = smiles_to_components([d["smiles"]], water_as_component1=True)
    if r.removed or not (np.isfinite(tg) and tg > 0): continue
    model = ActivityModel([WATER, Component(2, "o", r.components[1].subgroups)]); rows = []
    for T in (263.15, 283.15, 298.15, 313.15):
        try: eta0 = pure_organic_viscosity_vtf(T, tg)
        except Exception: continue
        for ww in np.linspace(0.98, 0.02, 15):
            try:
                res = model.evaluate([ww, 1 - ww], T, basis="mass"); aw = float(res.activity[0])
                eta = organic_mixture_viscosity(model, res.x, T, {2: eta0}).eta_pas
                if np.isfinite(aw) and 0 < aw <= 1.2 and np.isfinite(eta) and 0 < eta <= 1e12: rows.append((T, aw, np.log10(eta), ww))
            except Exception: pass
    if len(rows) < 10: continue
    o = featurize(d["name"], d["smiles"], "v")
    if o is None: continue
    o["tg"] = tg; vi = len(vm); vm.append(o)
    vr += [(vi,) + x for x in rows]
vr = np.array(vr); vpm = vr[:, 0].astype(int)
vcnt = np.array([counts(o) for o in vm]); vtest = np.array([hash100(o["smiles"]) < 15 for o in vm])
aT = np.column_stack([vr[:, 2], (vr[:, 1] - 293.15) / 20]).astype(np.float32); vy = vr[:, 3:4].astype(np.float32)
ltg = np.log([o["tg"] for o in vm]).astype(np.float32)[vpm][:, None]
print(f"viscosity chain: {len(vm)} molecules, {len(vr)} points")
# subsets for like-for-like comparisons (Sect. 3.6): (i) dry, near-glassy points (w_water <= 0.1 and
# T <= Tg + 50 K), the regime of the Tg-sensitivity analysis in notebook 04; (ii) the molecules whose 298 K curve
# admits a hyperbolic fit with R^2 >= 0.9 (the coefficient-model subset), at 298.15 K
from scipy.optimize import curve_fit
hyp = lambda a, p0, p1, p2: p0 + p1 / (a - p2)
good = set()
for vi in np.unique(vpm):
    sel = (vpm == vi) & (np.abs(vr[:, 1] - 298.15) < 0.01)
    if sel.sum() < 8: continue
    a, y = vr[sel, 2], vr[sel, 3]; o = np.argsort(a); a, y = a[o], y[o]
    try:
        p, _ = curve_fit(hyp, a, y, p0=[-3.0, 1.0, -0.5], bounds=([-20, -200, -5], [20, 200, a.min() - 0.05]), maxfev=5000)
        if 1 - np.sum((y - hyp(a, *p)) ** 2) / (np.sum((y - y.mean()) ** 2) + 1e-12) >= 0.9: good.add(vi)
    except Exception: pass
g = np.where(vtest[vpm])[0]
tgp = np.array([o["tg"] for o in vm])[vpm]
dry = (vr[g, 4] <= 0.1) & (vr[g, 1] - tgp[g] <= 50)
hsub = np.array([vpm[i] in good for i in g]) & (np.abs(vr[g, 1] - 298.15) < 0.01)
print(f"hyperbolic R²≥0.9 fits: {len(good)}/{len(vm)} molecules, {len({vpm[i] for i in g[hsub]})} of them held out; "
      f"dry near-glassy held-out points: {dry.sum()} of {len(g)}")
vis = []
for kind, xe in [("structure (C) only", aT), ("structure (C) + Tg", np.c_[aT, ltg])]:
    for seed in range(3):
        pred, _ = train_C(vcnt[vpm], xe, vy, np.where(~vtest[vpm])[0], seed, interp_split=True, max_epochs=400)
        e = np.abs(pred(vcnt[vpm[g]], xe[g]) - vy[g])[:, 0]
        vis.append({"inputs": kind, "seed": seed, "gen MAE": e.mean(), "gen MAE dry near-glassy": e[dry].mean(),
                    "gen MAE hyperbolic subset 298 K": e[hsub].mean()})
vis = pd.DataFrame(vis); vis.to_csv("C_viscosity_runs.csv", index=False)
display(vis.groupby("inputs")[["gen MAE", "gen MAE dry near-glassy", "gen MAE hyperbolic subset 298 K"]].agg(fmt))
print(f"total runtime {time.time()-T0:.0f}s")

viscosity chain: 352 molecules, 14895 points


hyperbolic R²≥0.9 fits: 123/352 molecules, 19 of them held out; dry near-glassy held-out points: 106 of 2200


,gen MAE,gen MAE dry near-glassy,gen MAE hyperbolic subset 298 K
inputs,,,
structure (C) + Tg,0.640 ± 0.056,1.669 ± 0.194,0.417 ± 0.052
structure (C) only,1.166 ± 0.068,2.657 ± 0.476,0.926 ± 0.119


total runtime 2185s
